# Baby Cry AI - Colab training

Runs the full supervised pipeline (YAMNet embeddings + MLP head) and produces a TFLite bundle for the Android app.

**Recommended:** Runtime -> Change runtime type -> **GPU** (optional; CPU also works, just slower).

Steps: (1) install, (2) optional dataset credentials, (3) train, (4) download the model bundle and drop it into the app.

In [ ]:
# 1) Get the code + install deps
REPO_URL = "https://github.com/mariostheot/baby-cry-analyzer"
# If the repo is PRIVATE, paste a GitHub token (fine-grained, Contents: Read) below.
# Leave it "" if the repo is public.
GITHUB_TOKEN = ""

import os
clone_url = REPO_URL
if GITHUB_TOKEN:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
if not os.path.exists("baby-cry-analyzer"):
    !git clone --depth 1 $clone_url
%cd baby-cry-analyzer/ml-training
!pip -q install -r requirements.txt
print("Setup done.")

## 2) Optional dataset credentials

Skip this entirely to train on the free **donateacry** corpus only. Provide credentials to unlock more data:
- **Kaggle** (balanced 8-class set): upload your `kaggle.json` (Kaggle -> Settings -> Create New Token).
- **Hugging Face** (CryCeleb2023, pretraining/gate): run `login()` and paste a token.

In [ ]:
# 2a) Kaggle credentials (optional) - upload kaggle.json, or press Cancel to skip
import os
try:
    from google.colab import files
    print("Upload kaggle.json (or Cancel to skip Kaggle):")
    uploaded = files.upload()
    if uploaded:
        os.makedirs("/root/.kaggle", exist_ok=True)
        with open("/root/.kaggle/kaggle.json", "wb") as fh:
            fh.write(list(uploaded.values())[0])
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
        print("Kaggle configured.")
except Exception as exc:
    print("Skipping Kaggle:", exc)

# 2b) Hugging Face (optional) - uncomment to enable CryCeleb2023
# from huggingface_hub import login
# login()

In [ ]:
# 3) Optional: enable the extra datasets you provided credentials for
import yaml
cfg_path = "configs/default.yaml"
cfg = yaml.safe_load(open(cfg_path))

# Uncomment the sources you unlocked above:
# cfg["datasets"]["kaggle_infant_cry"]["enabled"] = True
# cfg["datasets"]["infantcry_dbl"]["enabled"] = True
# cfg["datasets"]["cryceleb"]["enabled"] = True

yaml.safe_dump(cfg, open(cfg_path, "w"), sort_keys=False, allow_unicode=True)
print("classes:", cfg["classes"])
print("enabled datasets:", {k: v["enabled"] for k, v in cfg["datasets"].items()})

In [ ]:
# 4) Train: download -> manifest -> embed -> grouped CV -> evaluate -> export TFLite
!python -m src.main --config configs/default.yaml

In [ ]:
# 5) Inspect metrics and plots
import pathlib
from IPython.display import Image, display

art = pathlib.Path("artifacts")
if (art / "report.txt").exists():
    print((art / "report.txt").read_text())
for png in ["confusion_matrix.png", "roc_pr.png", "calibration.png"]:
    p = art / png
    if p.exists():
        display(Image(str(p)))

In [ ]:
# 6) Download the model bundle, then unzip it into app/src/main/assets/
#    (cry_reason.tflite, yamnet.tflite, labels.txt, cry_reason_trainable.tflite)
from google.colab import files
files.download("artifacts/model_bundle.zip")